In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

c:\dev\python\rag\rag_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from collections import Counter

In [3]:
file_path = "data/notes.pdf"

cleaning the repeated lines

In [4]:
def detect_repeated_lines(docs, threshold=0.6):
    """
    Lines appearing in more than `threshold` % of pages 
    are likely headers/footers
    """
    all_lines = []
    for doc in docs:
        lines = [line.strip() for line in doc.page_content.split('\n') 
                 if line.strip()]
        all_lines.extend(lines)
    
    total_pages = len(docs)
    line_counts = Counter(all_lines)
    
    # Flag lines appearing in 60%+ of pages
    repeated = {
        line for line, count in line_counts.items() 
        if count / total_pages >= threshold and len(line) > 3
    }
    
    return repeated

def clean_docs(docs):
    repeated_lines = detect_repeated_lines(docs)
    
    print("Detected repeating lines (will be removed):")
    for line in repeated_lines:
        print(f"  → '{line}'")
    
    for doc in docs:
        lines = doc.page_content.split('\n')
        cleaned = [l for l in lines if l.strip() not in repeated_lines]
        doc.page_content = '\n'.join(cleaned)
    
    return docs

In [5]:
loader = PyPDFLoader(file_path)
docs = loader.load()
docs = clean_docs(docs)

Detected repeating lines (will be removed):
  → 'Cyber Security[SEC-4]'
  → 'EPCHE, Department of BCA'


In [6]:
chunk_size=600
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=int(chunk_size * 0.10) )
chunks = splitter.split_documents(docs)

In [7]:
print(chunks[0].page_content)

1 
 
 
 
Module-I: Introduction to Cyber security 
Defining Cyberspace and Overview of Computer and Web -technology, Architecture of 
cyberspace, Communication and web technology, Internet, World wide web, Advent of 
internet, Internet infrastructure for data transfer and governance, Internet society, 
Regulation of cyberspace, Concept of cyber security, Issues and challenges of cyber 
security. 
 
Defining Cyberspace 
▪ The term Cyberspace was first coined by William Gibson in the year 1984. 
▪ Cyberspace is the environment in which communication over computer networks occurs.


In [8]:
len(chunks)

51

# Embeddings

In [9]:
embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=docs, 
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3278.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
query = "By whom the term cyberspace is coined?"

In [11]:
results_with_scores = vector_db.similarity_search_with_score(query, k=3)

for doc, score in results_with_scores:
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print('---')


Score: 0.6130
 1 
 
 
 
Module-I: Introduction to Cyber security 
Defining Cyberspace and Overview of Computer and Web -technology, Architecture of 
cyberspace, Communication and web technology, Internet, World wide web, Advent of 
internet, Internet infrastructure for data transfer and governance, Internet society, 
Regulation of cyberspace, Concept of cyber security, Issues and challenges of cyber 
security. 
 
Defining Cyberspace 
▪ The term Cyberspace was first coined by William Gibson in the year 1984. 
▪ Cyberspace is the environment in which communication over computer networks occurs. 
▪ Cyberspace is the virtual and dynamic space created by the machine clones. Cyberspace 
mainly refers to the computer which is a virtual network and is a medium electronically 
designed to help online communications to occur. 
▪ The primary purpose of creating cyberspace is to share information and communicate 
across the globe. 
▪ Cyberspace is that space in which users share information, inter

Checking model limit

In [12]:
from sentence_transformers import SentenceTransformer

# To get the value of the max sequence_length, we will query the underlying `SentenceTransformer` object used in the RecursiveCharacterTextSplitter
print(
    f"Model's maximum sequence length: {SentenceTransformer('all-MiniLM-L6-v2').max_seq_length}"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2494.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model's maximum sequence length: 256


# Retrevier

In [13]:
retriever = vector_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 4,
        'fetch_k': 25,       
        'lambda_mult': 0.5  
    }
)

checking the retriever is working

In [14]:
query = "By whom the term cyberspace is coined?"
def simple_retriever(retriever, query):
    retriever_docs = retriever.invoke(query)

    for i, doc in enumerate(retriever_docs) :
        print(f"--- Chunk {i+1} ---")
        print(doc)

simple_retriever(retriever=retriever, query= query)

--- Chunk 1 ---
page_content=' 1 
 
 
 
Module-I: Introduction to Cyber security 
Defining Cyberspace and Overview of Computer and Web -technology, Architecture of 
cyberspace, Communication and web technology, Internet, World wide web, Advent of 
internet, Internet infrastructure for data transfer and governance, Internet society, 
Regulation of cyberspace, Concept of cyber security, Issues and challenges of cyber 
security. 
 
Defining Cyberspace 
▪ The term Cyberspace was first coined by William Gibson in the year 1984. 
▪ Cyberspace is the environment in which communication over computer networks occurs. 
▪ Cyberspace is the virtual and dynamic space created by the machine clones. Cyberspace 
mainly refers to the computer which is a virtual network and is a medium electronically 
designed to help online communications to occur. 
▪ The primary purpose of creating cyberspace is to share information and communicate 
across the globe. 
▪ Cyberspace is that space in which users share in

Let's change the query - the question is from middle 

In [15]:
query = "what is Internet infrastructure for data transfer and governance "

simple_retriever(retriever=retriever, query= query)

--- Chunk 1 ---
page_content=' 7 
 
 
Institutionalization Phase (1975 to 1995) 
− large institutions such as the U.S. Department of Defense (DoD) and the National 
Science Foundation (NSF) provided funding and legitimization for the fledging 
Internet. 
Commercialization Phase (1995 to the present) 
− The U.S. government encouraged private corporations to take over and expand the 
Internet backbone as well as local service beyond military installations and college 
campuses to the rest of the population around the world. 
Internet infrastructure for data transfer and governance 
▪ Internet infrastructure for data transfer and governance encompasses the physical and 
virtual systems, protocols, and regulations that enable the secure, efficient, and reliable 
exchange of data across the global network. 
▪ This infrastructure plays a critical role in ensuring data privacy, security, and compliance with 
regulations. 
▪ Here are key components and considerations for internet infrastructur

# It's time for Retrieval chain

In [16]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import ChatPromptTemplate


In [17]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

In [18]:
llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free"
)

In [22]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question between 4-5 sentences."
    "\n\n"
    "The output format should be in neat with break for each sentence"
    "User '▪' for each points"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm= llm, prompt= prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": query})
print(response["answer"])


▪ Internet infrastructure for data transfer and governance encompasses the physical and virtual systems, protocols, and regulations that enable the secure, efficient, and reliable exchange of data across the global network.

▪ This infrastructure plays a critical role in ensuring data privacy, security, and compliance with regulations through various components including network infrastructure, protocols, and data centers.

▪ Key elements of this infrastructure include backbone networks, last-mile connectivity, data centers, and protocols like IP, TLS, HTTP/HTTPS, and DNSSEC that facilitate proper data routing and security.

▪ Governance aspects involve compliance with data privacy regulations like GDPR and CCPA, access controls, encryption standards, and oversight by bodies such as ICANN that manage domain name systems and policies.
